In [ ]:
import httpx
import pathlib
from decouple import config

In [ ]:
NBS_DIR = pathlib.Path().resolve()
REPO_DIR = NBS_DIR.parent
DATA_DIR = REPO_DIR / "data"
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True, parents=True)


In [ ]:
API_ACCESS_KEY = config('API_ACCESS_KEY')

headers = {
    "X-API-Key": API_ACCESS_KEY
}
endpoint = "http://127.0.0.1:8000/predictions"
preds_res = httpx.get(endpoint, 
                params={"status": "succeeded"},
                headers=headers)
preds_json = preds_res.json()
# preds_json

In [ ]:
BASE_URL="http://127.0.0.1:8000"
for pred in preds_json:
    path = pred.get('url')
    endpoint = f"{BASE_URL}{path}"
    res = httpx.get(endpoint, headers=headers)
    if res.status_code not in range(200, 299):
        continue
    data = res.json()
    files = data.get('files') or None
    if files is None:
        continue
    obj_id = data.get('id')
    with httpx.Client() as client:
        for i, file_path in enumerate(files):
            fname = pathlib.Path(file_path).name
            outpath = GENERATED_DIR / obj_id / fname
            outpath.parent.mkdir(exist_ok=True, parents=True)
            if outpath.exists():
                continue
            url = f"{BASE_URL}{file_path}"
            res = client.get(url, headers=headers)
            res.raise_for_status()
            with open(outpath, 'wb') as f:
                f.write(res.content)